In [16]:
year_month="2024-03"

StatementMeta(, 750a02d0-f8b4-439c-9d51-6a3244b50e74, 20, Finished, Available, Finished, False)

In [17]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

for s in ["silver","ops"]:
    spark.sql(f"Create SCHEMA IF Not Exists {s}")

def upsert(df,table,key):
    if spark.catalog.tableExists(table):
        (DeltaTable.forName(spark,table).alias("t")
        .merge(df.alias("s"),f"t.{key}=s.{key}")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

    else:
        df.write.format("delta").saveAsTable(table)

def log_run(step, rows_in, rows_good, rows_rejected, status, message=""):
    cols=["step","year_month","rows_in","rows_good","rows_rejected","status","message"]
    row=[(step,year_month,int(rows_in),int(rows_good),int(rows_rejected),status,message,datetime.now())]
    spark.createDataFrame(row,cols).write.mode("append").format("delta").saveAsTable("ops.pipeline_log")

StatementMeta(, 750a02d0-f8b4-439c-9d51-6a3244b50e74, 21, Finished, Available, Finished, False)

In [18]:
try:
    raw=spark.read.parquet(f"Files/Bronze/taxi/yellow_tripdata_{year_month}.parquet")
    rows_in=raw.count()
  
    df=(raw.select(
            F.col("VendorID").cast("int").alias("vendor_id"),
            F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_ts"),
            F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_ts"),
            F.col("passenger_count").cast("int").alias("passenger_count"),
            F.col("trip_distance").cast("double").alias("trip_distance"),
            F.col("PULocationID").cast("int").alias("pu_location_id"),
            F.col("DOLocationID").cast("int").alias("do_location_id"),
            F.col("payment_type").cast("int").alias("payment_type"),
            F.col("fare_amount").cast("double").alias("fare_amount"),
            F.col("tip_amount").cast("double").alias("tip_amount"),
            F.col("total_amount").cast("double").alias("total_amount"))
            .withColumn("year_month",F.lit(year_month))
            .withColumn("duration_min",F.unix_timestamp("dropoff_ts")-F.unix_timestamp("pickup_ts")/60))


     # 2. Data quality rules, first failing rule becomes the reject reason
    df = df.withColumn("reject_reason",
        F.when(F.col("pickup_ts").isNull() | F.col("dropoff_ts").isNull(), "null_timestamp")
         .when(F.col("dropoff_ts") < F.col("pickup_ts"), "dropoff_before_pickup")
         .when(F.date_format("pickup_ts", "yyyy-MM") != F.col("year_month"), "pickup_outside_month")
         .when(F.col("fare_amount") <= 0, "non_positive_fare")
         .when((F.col("trip_distance") <= 0) | (F.col("trip_distance") > 100), "bad_distance")
         .when(F.col("duration_min") > 360, "trip_over_6_hours")
         .when(F.col("pu_location_id").isNull() | F.col("do_location_id").isNull(), "missing_zone"))
    
    good=df.filter(F.col("reject_reason").isNull()).drop("reject_reason")
    bad=df.filter(F.col("reject_reason").isNotNull()).withColumn("ingest_ts",F.current_timestamp())

    good=(good
        .withColumn("trip_duration_min",F.round("duration_min",2)).drop("duration_min")
        .withColumn("pickup_hours",F.date_trunc("hour","pickup_ts"))
        .withColumn("tip_pct",F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2))
        .withColumn("trip_id", F.sha2(F.concat_ws("|", "vendor_id", "pickup_ts", "dropoff_ts","pu_location_id", "do_location_id", "total_amount"), 256))
        .dropDuplicates(["trip_id"])
        .withColumn("ingest_ts", F.current_timestamp()))

    if spark.catalog.tableExists("ops.dq_rejects"):
        spark.sql(f"DELETE From ops.dq_rejects Where year_month='{year_month}'")
    bad.write.mode("append").format("delta").saveAsTable("ops.dq_rejects")
    upsert(good,"silver.taxi_trips","trip_id")


    log_run("taxi_silver",rows_in,good.count(),bad.count(),"SUCCESS")
except Exception as e:
    log_run("taxi_silver",0,0,0,"FAILED",str(e)[:500])

StatementMeta(, 750a02d0-f8b4-439c-9d51-6a3244b50e74, 22, Finished, Available, Finished, False)

In [19]:
try:
    w = spark.read.option("multiLine", True).json(f"Files/Bronze/weather/weather_{year_month}.json")
    z = F.arrays_zip(
        F.col("hourly.time").alias("time"),
        F.col("hourly.temperature_2m").alias("temp"),
        F.col("hourly.precipitation").alias("precip"),
        F.col("hourly.wind_speed_10m").alias("wind"))
    weather = (w.select(F.explode(z).alias("r"))
        .select(F.to_timestamp("r.time").alias("weather_hour"),
                F.col("r.temp").cast("double").alias("temp_c"),
                F.col("r.precip").cast("double").alias("precipitation_mm"),
                F.col("r.wind").cast("double").alias("wind_kmh")))
    upsert(weather, "silver.weather_hourly", "weather_hour")

    zones = (spark.read.option("header", True).csv("Files/Bronze/zones/taxi_zone_lookup.csv")
        .select(F.col("LocationID").cast("int").alias("location_id"),
                F.col("Borough").alias("borough"),
                F.col("Zone").alias("zone"),
                F.col("service_zone")))
    zones.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver.zones")

    log_run("weather_zones_silver", weather.count(), weather.count(), 0, "SUCCESS")
except Exception as e:
    log_run("weather_zones_silver", 0, 0, 0, "FAILED", str(e)[:500])
    raise

StatementMeta(, 750a02d0-f8b4-439c-9d51-6a3244b50e74, 23, Finished, Available, Finished, False)